In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import json
import time
import cv2
import pywt
import wfdb
import numpy as np
import tensorflow as tf
from datetime import datetime
from scipy.signal import resample
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from cnn2snn import quantize, convert, load_quantized_model
import random

from data import ECGDatasetBuilder,ECGDatasetLoader
from model import build_akida_model, prepare_qat_model, apply_activity_regularizer

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# Directory Setup
DATA_DIR = "../processed_data"
RUN_DIR_BASE = "../akida_ecg_scalogram"
RAW_DATA_DIR = "../data//mitdb"  

# Training Configuration
LEARNING_RATE = 3e-3
FLOAT_EPOCHS = 80
QAT_EPOCHS = 50
BATCH_SIZE = 64
INPUT_SHAPE = (36, 32, 1)
NUM_CLASSES = 3
L1L2_REG_VALUE = 2e-5

SEED = 67004546
# Apply it everywhere
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print(f"Training Seed = {SEED}")

# Class weights for handling class imbalance
CLASS_WEIGHTS = {0: 1.0, 1: 6.0, 2: 3.0}

# Dataset Records Definition
TRAIN_RECORDS = [
    '101', '106', '108', '109', '112', '114', '115', '116',
    '118', '119', '122', '124', '201', '203', '205', '207',
    '208', '209', '215', '220', '223', '230'
]

TEST_RECORDS = [
    '100', '103', '105', '111', '113', '117', '121', '123',
    '200', '202', '210', '212', '213', '214', '219', '221',
    '222', '228', '231', '232', '233', '234'
]

# Create Run Directory with Timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(RUN_DIR_BASE, f"3class_arrhythmia_classification_{timestamp}")
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Run Directory created at: {RUN_DIR}")

In [ ]:
# ==============================================================================
# EVALUATION & METRICS METRIC PLOTTERS
# ==============================================================================
def evaluate_and_report(model, X_test, y_test, run_dir, filename_suffix=""):
    """Generates classification metrics report and logs it to a text file."""
    preds = np.argmax(model.predict(X_test), axis=1)
    report = classification_report(y_test, preds, target_names=["N", "S", "V"])
    print(f"\n--- Classification Report {filename_suffix} ---")
    print(report)
    
    report_path = os.path.join(run_dir, f"classification_report{filename_suffix}.txt")
    with open(report_path, "w") as f:
        f.write(report)

In [ ]:
# Process Raw Data if specified, otherwise load cached numpy arrays
if RAW_DATA_DIR and os.path.exists(RAW_DATA_DIR):
    print("Preprocessing raw MIT-BIH dataset...")
    builder = ECGDatasetBuilder(
        dataset_path=RAW_DATA_DIR, 
        data_dir=DATA_DIR, 
        window_before=250, 
        window_after=250, 
        img_size=32
    )
    builder.preprocess_dataset(TRAIN_RECORDS, TEST_RECORDS)
else:
    print("Skipping raw preprocessing, loading from preprocessed files...")

# Load Datasets
loader = ECGDatasetLoader(data_dir=DATA_DIR)
X_train, y_train, X_val, y_val, X_test, y_test = loader.load_dataset()

In [ ]:
print("\n--- Phase 1: Training FP32 Baseline Model ---")

# Build and regularize FP32 model
model = build_akida_model(INPUT_SHAPE, NUM_CLASSES, L1L2_REG_VALUE)
apply_activity_regularizer(model, L1L2_REG_VALUE)

model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

fp32_path = os.path.join(RUN_DIR, "arrhythmia_classification_fp32_model.h5")
fp32_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(filepath=fp32_path, monitor="val_loss", save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, verbose=2),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=False)
]

history = model.fit(
    X_train, y_train, 
    validation_data=(X_val, y_val),
    epochs=FLOAT_EPOCHS, 
    batch_size=BATCH_SIZE, 
    class_weight=CLASS_WEIGHTS,
    callbacks=fp32_callbacks
)

# Export logs and last epoch model
with open(os.path.join(RUN_DIR, "training_log.json"), "w") as f:
    json.dump({k: [float(x) for x in v] for k, v in history.history.items()}, f, indent=4)
model.save(os.path.join(RUN_DIR, "arrhythmia_classification_fp32_last_model.h5"))

# Evaluate Best FP32 Model
print("\n--- Phase 1: Evaluating FP32 Baseline Model ---")
best_fp32 = tf.keras.models.load_model(fp32_path)
evaluate_and_report(best_fp32, X_test, y_test, RUN_DIR, filename_suffix="")

In [ ]:
print("\n--- Phase 2: Starting Quantization Aware Training (QAT) ---")

# Prepare layer node mapping and QAT Graph Transformation
best_fp32.input_names = [tensor.name.split(":")[0] for tensor in best_fp32.inputs]
q_model = prepare_qat_model(best_fp32)

q_model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

qat_path = os.path.join(RUN_DIR, "arrhythmia_classification_qat_model.h5")
qat_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(filepath=qat_path, monitor="val_loss", save_best_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, verbose=2),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=False)
]

q_history = q_model.fit(
    X_train, y_train, 
    validation_data=(X_val, y_val),
    epochs=QAT_EPOCHS, 
    batch_size=BATCH_SIZE, 
    class_weight=CLASS_WEIGHTS,
    callbacks=qat_callbacks,
    verbose=1
)

# Export QAT training logs and last epoch model
with open(os.path.join(RUN_DIR, "arrhythmia_classification_qat_training_log.json"), "w") as f:
    json.dump({k: [float(x) for x in v] for k, v in q_history.history.items()}, f, indent=4)
q_model.save(os.path.join(RUN_DIR, "arrhythmia_classification_qat_last_model.h5"))

# Evaluate QAT Model
print("\n--- Phase 2: Evaluating QAT Model ---")
best_qat = load_quantized_model(qat_path)
evaluate_and_report(best_qat, X_test, y_test, RUN_DIR, filename_suffix="_quant")

print(f"\n[SUCCESS] Pipeline execution finished. All artifacts saved to: {RUN_DIR}")